# SatClip: LoRA fine-tuning of the language layer (Colab)

This notebook trains the part of SatClip that reads questions and explains answers. It never trains anything that produces a number: water area, change and crop decline come from the classical instruments in `backend/satclip/instruments/`, and their confidence comes from the calibration in `training/calibration/`.

What you get at the end:
1. a LoRA adapter for **Qwen2-VL-2B-Instruct** that turns English, Hinglish and Hindi questions with any Indian date style into SatClip's intent JSON;
2. the same adapter trained on optical (RSVQA) and optionally radar plus optical (BigEarthNet) Q&A, so it can talk about the image it shows, and refuses to count;
3. evaluation reports next to the rule-parser baseline, in `training/eval/reports/`.

Runtime: **GPU (T4 is enough with QLoRA)**. Menu: Runtime, Change runtime type, T4 GPU. A full run of the default settings takes roughly an hour on a T4 (estimate, not measured).

In [ ]:
!git clone --depth 1 https://github.com/UnKnownnPasta/satclip.git
%cd satclip
!pip -q install -e "backend[geo]" "transformers>=4.49" peft accelerate bitsandbytes pandas pyarrow matplotlib
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Build the training data
All three builders are deterministic (fixed seeds). The intent set splits districts 80/10/10, so the test set only has districts the model never saw.

In [ ]:
!python training/lora/build_intent_data.py --n-train 12000
!python training/lora/build_rsvqa.py --download --split train --max 20000
!python training/lora/build_rsvqa.py --split test --max 2000

### Optional: radar and optical Q&A from BigEarthNet v2
The full archives are 63 GB (S2) and 54 GB (S1). Extract a few Sentinel tiles only. Skip this cell for a first run.

In [ ]:
RUN_BIGEARTHNET = False
if RUN_BIGEARTHNET:
    !mkdir -p /content/ben && cd /content/ben && wget -q -c "https://zenodo.org/records/10891137/files/BigEarthNet-S2.tar.zst?download=1" -O S2.tar.zst
    !cd /content/ben && tar --zstd -xf S2.tar.zst --wildcards '*T33UUP*' && rm S2.tar.zst
    !python training/lora/build_bigearthnet.py --s2-root /content/ben/BigEarthNet-S2 --max 5000

## 2. Baseline before training
The shipped rule parser on the same held-out questions. Any adapter has to beat this.

In [ ]:
!python training/eval/eval_intents.py --predictor rules
!python training/eval/eval_vqa.py --set rsvqa_lr --predictor majority

## 3. Train (QLoRA, rank 16, language model only)

In [ ]:
DATA = "training/data/intents/train.jsonl training/data/rsvqa_lr/train.jsonl"
import os
if os.path.exists("training/data/bigearthnet/train.jsonl"):
    DATA += " training/data/bigearthnet/train.jsonl"
!python training/lora/train_lora.py --data $DATA --max-per-file 12000 --qlora --epochs 1 --batch 4 --grad-accum 4 --out training/runs/satclip-lora

## 4. Evaluate the adapter

In [ ]:
!python training/eval/eval_intents.py --predictor hf --adapter training/runs/satclip-lora --name qwen2vl2b_lora
!python training/eval/eval_vqa.py --set rsvqa_lr --predictor hf --adapter training/runs/satclip-lora --name rsvqa_qwen2vl2b_lora
!cat training/eval/reports/intents_rules.md training/eval/reports/intents_qwen2vl2b_lora.md

## 5. Refit the water calibration (CPU is fine)
Streams the 446 Sen1Floods11 hand-labelled chips and refits `config/calibration/sar_water_otsu.json`.

In [ ]:
!python training/calibration/fit_water.py --workers 8
!cat training/calibration/reports/sar_water_otsu.md

## 6. Keep the results
Copy the adapter and reports to Google Drive, or commit the reports back to the repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/satclip && cp -r training/runs/satclip-lora training/eval/reports /content/drive/MyDrive/satclip/